In [1]:
from pathlib import Path

import pandas as pd
import yfinance as yf

# Build Pair-Divergence Market Data

Download raw daily OHLCV data for the pair-divergence stock universe.

In [2]:
currentDirectory = Path.cwd()

possibleRoots = [
    currentDirectory,
    currentDirectory.parent,
    currentDirectory.parent.parent
]

repositoryRoot = None

for directory in possibleRoots:
    universePath = directory / "data" / "universes" / "pairDivergenceUniverse.txt"

    if universePath.exists():
        repositoryRoot = directory
        break

if repositoryRoot is None:
    raise FileNotFoundError("Could not find pairDivergenceUniverse.txt")

universePath = repositoryRoot / "data" / "universes" / "pairDivergenceUniverse.txt"
marketDataDirectory = repositoryRoot / "data" / "pairDivergence"
outputPath = marketDataDirectory / "pairDivergenceData_Raw.parquet"

## Load Universe

In [3]:
symbols = pd.read_csv(
    universePath,
    header=None,
    names=["symbol"],
    dtype=str,
    keep_default_na=False
)["symbol"].str.strip()

symbols = symbols[symbols != ""].drop_duplicates().tolist()

print(f"Universe symbols: {len(symbols):,}")

Universe symbols: 5,289


## Download Raw Market Data

In [4]:
startDate = "2018-01-01"
endDate = "2026-07-01"
batchSize = 100

print(f"Start date: {startDate}")
print(f"End date: {endDate} (exclusive)")
print(f"Batch size: {batchSize}")

Start date: 2018-01-01
End date: 2026-07-01 (exclusive)
Batch size: 100


In [5]:
priceFrames = []
failedSymbols = []

requiredPriceColumns = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
batchCount = (len(symbols) + batchSize - 1) // batchSize

for batchIndex in range(batchCount):
    batchStart = batchIndex * batchSize
    symbolBatch = symbols[batchStart:batchStart + batchSize]

    print(f"Downloading batch {batchIndex + 1}/{batchCount}")

    batchData = yf.download(
        symbolBatch,
        start=startDate,
        end=endDate,
        interval="1d",
        auto_adjust=False,
        actions=False,
        repair=True,
        keepna=False,
        group_by="ticker",
        threads=True,
        progress=False,
        ignore_tz=True,
        multi_level_index=True
    )

    if batchData is None or batchData.empty:
        failedSymbols.extend(symbolBatch)
        continue

    if not isinstance(batchData.columns, pd.MultiIndex):
        raise ValueError("Unexpected Yahoo column structure")

    availableSymbols = set(batchData.columns.get_level_values(0))

    for symbol in symbolBatch:
        if symbol not in availableSymbols:
            failedSymbols.append(symbol)
            continue

        symbolData = batchData[symbol].copy()

        if not set(requiredPriceColumns).issubset(symbolData.columns):
            failedSymbols.append(symbol)
            continue

        symbolData = symbolData[requiredPriceColumns].dropna(how="all")

        if symbolData.empty:
            failedSymbols.append(symbol)
            continue

        symbolData = symbolData.reset_index()
        symbolData["symbol"] = symbol

        symbolData = symbolData.rename(columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adjClose",
            "Volume": "volume"
        })

        symbolData = symbolData[
            ["date", "symbol", "open", "high", "low", "close", "adjClose", "volume"]
        ]

        priceFrames.append(symbolData)

rawMarketData = pd.concat(priceFrames, ignore_index=True)
rawMarketData = rawMarketData.sort_values(["symbol", "date"]).reset_index(drop=True)

print()
print(f"Downloaded symbols: {rawMarketData['symbol'].nunique():,}")
print(f"Rows: {len(rawMarketData):,}")
print(f"Initial failures: {len(failedSymbols):,}")

if failedSymbols:
    print("Failed symbols:")
    print(failedSymbols)

$ADBT: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")
$ADIG: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")

3 Failed downloads:
['ADBT', 'ADIG']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")
['AAUC']: possibly delisted; no price data found  (1h 2025-05-20 -> 2025-05-29)


$APMD: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")
$APPS: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01)

2 Failed downloads:
['APMD']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")
['APPS']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01)


$ATTO: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")

1 Failed download:
['ATTO']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")



1 Failed download:
['BESS']: possibly delisted; no price data found  (1h 2025-01-30 -> 2025-02-07)


$BIOT: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")
$BLSM: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")

2 Failed downloads:
['BIOT', 'BLSM']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")


...

$CIGL: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")

1 Failed download:
['CIGL']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")


$COPR: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")

1 Failed download:
['COPR']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")


$CSQR: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")

1 Failed download:
['CSQR']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-01) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1782878400")


$DCX: possibly delisted; no price data found  (1h 2025-04-16 -> 2025-07-09)



Downloaded symbols: 5,234
Rows: 8,835,632
Initial failures: 55
Failed symbols:
['ADBT', 'ADIG', 'APMD', 'APPS', 'ATTO', 'BIOT', 'BLSM', 'BRVE', 'BSP', 'CCCT', 'CEPL', 'CGCF', 'CIGL', 'COPR', 'CSQR', 'FBDT', 'FOIL', 'GHXI', 'GIXI', 'GYGY', 'IMC', 'IOND', 'IPVV', 'ITG', 'JMKE', 'JONE', 'LIME', 'LTGO', 'LTGR', 'LYNX', 'NUR', 'OBX', 'PHOS', 'QMLS', 'QVCG', 'RACD', 'REF', 'REFI', 'SCTX', 'SECZ', 'SGLD', 'SHOT', 'SKHY', 'SNSC', 'STDN', 'SVA', 'TP', 'TRBG', 'TREO', 'VCRE', 'VOGX', 'WLCO', 'XTND', 'YFOR', 'YICC']


## Validate Raw Market Data

In [6]:
expectedColumns = ["date", "symbol", "open", "high", "low", "close", "adjClose", "volume"]

if rawMarketData.columns.tolist() != expectedColumns:
    raise ValueError(f"Unexpected columns: {rawMarketData.columns.tolist()}")

duplicateCount = rawMarketData.duplicated(subset=["date", "symbol"]).sum()
missingValueCounts = rawMarketData[["open", "high", "low", "close", "adjClose", "volume"]].isna().sum()
nonpositivePriceCount = (
    (rawMarketData[["open", "high", "low", "close"]] <= 0)
    .any(axis=1)
    .sum()
)

negativeVolumeCount = (rawMarketData["volume"] < 0).sum()

invalidHighCount = (
    (rawMarketData["high"] < rawMarketData["open"]) |
    (rawMarketData["high"] < rawMarketData["low"]) |
    (rawMarketData["high"] < rawMarketData["close"])
).sum()

invalidLowCount = (
    (rawMarketData["low"] > rawMarketData["open"]) |
    (rawMarketData["low"] > rawMarketData["high"]) |
    (rawMarketData["low"] > rawMarketData["close"])
).sum()

print(f"Rows: {len(rawMarketData):,}")
print(f"Symbols: {rawMarketData['symbol'].nunique():,}")
print(f"First date: {rawMarketData['date'].min().date()}")
print(f"Last date: {rawMarketData['date'].max().date()}")
print(f"Duplicate date-symbol rows: {duplicateCount:,}")
print(f"Nonpositive price rows: {nonpositivePriceCount:,}")
print(f"Negative volume rows: {negativeVolumeCount:,}")
print(f"Invalid high rows: {invalidHighCount:,}")
print(f"Invalid low rows: {invalidLowCount:,}")
print()
print("Missing values:")
print(missingValueCounts.to_string())

Rows: 8,835,632
Symbols: 5,234
First date: 2018-01-02
Last date: 2026-06-30
Duplicate date-symbol rows: 0
Nonpositive price rows: 2
Negative volume rows: 0
Invalid high rows: 1
Invalid low rows: 6

Missing values:
Price
open          0
high          0
low           0
close         0
adjClose    732
volume        0


In [7]:
# Identify raw OHLC anomalies
rawAnomalyMask = (
    (rawMarketData[["open", "high", "low", "close"]] <= 0).any(axis=1) |
    (rawMarketData["high"] < rawMarketData["open"]) |
    (rawMarketData["high"] < rawMarketData["low"]) |
    (rawMarketData["high"] < rawMarketData["close"]) |
    (rawMarketData["low"] > rawMarketData["open"]) |
    (rawMarketData["low"] > rawMarketData["high"]) |
    (rawMarketData["low"] > rawMarketData["close"])
)

rawAnomalies = rawMarketData.loc[rawAnomalyMask].copy()

print(f"Unique anomalous rows: {len(rawAnomalies):,}")

display(rawAnomalies)

Unique anomalous rows: 9


Price,date,symbol,open,high,low,close,adjClose,volume
1673885,2021-05-05,CIG-C,1.532544,1.609467,1.538461,1.609467,1.051378,22722.0
1827514,2021-10-08,CNL,1.205000,2.410000,0.000000,1.205000,1.205000,500.0
3840331,2021-05-05,HUBB,195.979996,198.639999,196.210007,198.130005,181.720032,127234.0
7086847,2021-05-05,SHEL,39.060001,39.839901,39.445000,39.779999,32.394276,3225346.0
7087280,2023-01-24,SHEL,51.259998,58.400002,56.259998,57.990002,50.033306,3125185.0
7143563,2021-05-05,SITC,17.696529,17.529917,17.125290,17.137190,4.175779,818516.0
7357196,2021-09-29,SPRC,24570.000000,26535.599609,0.000000,26535.599609,26535.599609,1.0
8029891,2021-05-05,UA,20.870001,21.825001,21.000000,21.129999,21.129999,5213105.0
8547910,2021-05-05,WLY,59.810001,61.299999,59.950001,61.200001,51.322136,261515.0


## Save Raw Market Data

In [8]:
# Remove Yahoo column metadata
rawMarketData.columns.name = None

# Save raw OHLCV data
marketDataDirectory.mkdir(parents=True, exist_ok=True)
rawMarketData.to_parquet(outputPath, index=False)

print(f"Rows saved: {len(rawMarketData):,}")
print(f"Symbols saved: {rawMarketData['symbol'].nunique():,}")
print(f"Saved to: {outputPath}")

Rows saved: 8,835,632
Symbols saved: 5,234
Saved to: /Users/suhas/Documents/github/Quant-Research-Lab/data/pairDivergence/pairDivergenceData_Raw.parquet
